# Making drugs in silico - Part III

Now that you know how to equilibrate model membranes, and how to modify the topology of some simple molecules to fit the physico-chemical conditions in their respective environment, we can apply this knowledge to a real case.

<a href="https://en.wikipedia.org/wiki/Cisplatin">Cisplatin</a> is one the most used antineoplastics at the day. However, it is known to be poorly selective to cancer cells, driving to severe side effects for the patients. Its transport to the cytoplasm, where it is going to induce cell death by competing pathways, is mediated by the plasma membrane of the cell. At the time, it is presumed that the **major mode of transport** occurs through the lipid matrix of the plasma membrane and is thus only a **simple passive diffusion** process. Cisplatin being a hydrophilic molecule, this process is not favorable due to the presence of lipid tails. If we can model this process with MD simulations and obtain informations about the kinetics of that transport, we could confirm that cisplatin really permeates passively through the membrane. We also could measure how the lipid content in the plasma membrane affects the diffusion and discuss valuable questions on cisplatin selectivity.

Hence, **our objective** during the two following sessions is to understand how to access such properties, and particularly, how **to compute the permeability coefficient** of cisplatin facing models of the lipid matrix of a sane and cancer plasma membranes.

## 1. A bit of theory

### 1.1. Permeability coefficient

One fundamental element to assess drug's efficiency is linked to its kinetics. In case of drugs that need to permeate through the plasma membrane to act intracellularly, one option is to create a pharmocokinetic model. In the case of cisplatin, one would like to estimate the evolution of the quantity of native and solvated cisplatin depending on the salt content and pH of the medium. A pharmacokinetic model takes the form of a set of differential equations piloted by kinetic constants. The equations are often simplified to be in the case of simple 1st order chemical reactions.

The permeability coefficient P of a solute through a lipid membrane, is a coefficient closely linked to the kinetic constant. Indeed, the effective kinetic constant mainly differs by taking into account the shape of the membrane (such as a spheroid).

### 1.2. Inhomogeneous solubility-diffusion model

Based on experiments, the Meyer-Overton's rule drove to write the so-called **homogeneous solubility diffusion** model. This model depicts a lipid membrane as an homogeneous oil slab through which a small solute permeates. In this case, the permeability coefficient is simply related to the <a href="https://en.wikipedia.org/wiki/Octanol-water_partition_coefficient">oil/solvent partition coefficient K</a>, the thickness of the slab d, and the diffusion coefficient D of the solute in the oil slab:

$$ P = \frac {KD} {d} $$

However, it happens that lipid membranes are very much heterogeneous. In the case of a one-component membrane, the heterogeneity mainly resids along the normal to the membrane (for us, the Z axis). Hence, the inhomogeneous solubility-diffusion (ISD) model was proposed:

$$ \frac {1} {P} = \int_{-d/2}^{d/2}  \frac {1} {K(Z)D(Z)} dZ = \int_{-d/2}^{d/2} \frac { e^{ \beta \Delta G(Z) } } {D(Z)} dZ $$,

where $\beta = \frac {1} {k_{B}T} $, and the position-dependent <a href="https://en.wikipedia.org/wiki/Partition_function_(statistical_mechanics)">partition coefficient</a> is directly expressed as a free energy $K = e^{- \beta \Delta G(Z) }$.

Hence, during these two sessions, we want to compute both $\Delta G(Z)$ and $D(Z)$ for Z between -d/2 and +d/2.

### 1.3. How to compute the free energy of the system?

In MD simulations, the measure of the potential energy of the system is inherent to the simulation as it is computed at every frame while propagating the system through time by integrating the equations of motion. The potential energy represents the energy associated with a specific conformation – or microstate – of the system.

In a macroscopic thermodynamic state, there are numerous microstates or conformations that correspond to different arrangements of the system's atoms and molecules. Accessing the free energy of a system that transitions between different states requires formally sampling each microstate associated with a given macroscopic state. This is because in statistical physics, to obtain meaningful measurements, one must perform an ensemble average, which involves sampling all accessible conformations.

MD simulations allow for the exploration of different conformations by generating trajectories that capture the system's evolution over time. However, it is important to recognize that MD simulations, in their pristine form, may not adequately sample the entire conformational space or sufficiently explore high-energy regions and energy barriers. The limitations of sampling arise due to the finite length of simulations and the inherent stochasticity of molecular dynamics.

To better understand the origin of the sampling difficulties that arise to measure the free energy using pristine MD simulations, it is best to leave the thermodynamic point of view and to further refer to statistical physics. In this frame, in the case of the canonical ensemble, the probability $P_i$ that the system lies in the microstate $i$ is such that $P_i \propto e^{-\beta E_i}$ which means that it decreases exponentially with the energy of the system in the given microstate $i$. This observation further explains why natural MD simulations will undersample regions of high energy. It also explains that the system will potentially never cross high energy barriers during the course of a simulation, driving to potentially unexplored portions of the RC.

Hence, the community gradually developped many techniques of so-called enhanced sampling. During these two sessions, we will dig into one of them, called umbrella sampling.

### 1.4. Reaction coordinates, collective variables, and free energy

To obtain meaningful free energy profiles of a given process, this one should be evaluated along the minimum free energy path (MFEP) on the free energy surface (FES). This path is formally described by a basis formed by a set of orthogonal vectors referred to as reaction coordinates (RCs).

When computing the free energy profile of a process, one must identify and estimate this set of reaction coordinates. However, directly sampling all the degrees of freedom of a system is generally not accessible either because of the high dimension of subsequent RCs, or most often, because we lack the knowledge to appropriately describe these RCs formally. Collective variables are functions that capture the essential features of a system and provide a low-dimensional representation of the reaction coordinates. Formally that has two consequences that one tries to minimize during their design: 
1. the basis they generate is incomplete, but to make the CVs meaninful, the projection of the MFEP on the CVs should account for most of it,
2. the basis might not be generated by orthogonal vectors, which leads to double-counting issues, but the choice of the CVs is generally made such that orthogonality is mainly preserved. 

These variables are typically chosen based on their relevance to the process under investigation. They can include geometric parameters, distances between atoms or groups, angles, torsional angles, or any other suitable descriptors that capture the important changes occurring during the process. Employing accurate CVs can allow to effectively navigate the high-dimensional configuration space and to focus on the essential changes that define the reaction coordinates. The choice of appropriate collective variables is crucial for obtaining accurate and informative free energy profiles.

## 2. Downloading the files and setting the environment

As usual, let's download the Jupyter notebook and set up the environment. Since we have a little bit of data, it might be safer this week to download them to the <code>/scratch</code> partition for which the quota is not limited as for the <code>home</code> partition. However, mind that data on that partition can be deleted without prior notice.

```bash
cd /scratch/$USER
git clone https://gitlab.com/denysbiriukov/survival_guide_simulations_c9555.git
cd survival_guide_simulations_c9555.git/week07
bash all.sh
```


## 3. Creating the membrane-drug initial system

You can now browse through the file system of this practical and go to the first part: <code>01.build/01.input</code>.

In [ ]:
# Browse to the correct directory
%cd 

The plan of this session is to build a system to compute the free energy profile of permeation, which will allow us to access the permeability coefficient of native cisplatin through a model of the lipid matrix of the plasma membrane. For that, we need to build a system that holds cisplatin molecules in the solvent above some model membrane.

The model membrane was equilibrated beforehand (<code>membrane.gro</code> and <code>membrane.tpr</code>), so that we only need to use these files to place the cisplatin molecules in the solvent. Many strategies can be undertaken to achieve this, for instance:
* Using <code>gmx insert-molecules</code>,
* Selecting only the lipid membrane, solvating the cisplatin using <i>e.g.</i> <code>gmx solvate</code>, and merging the two selections / files

Here, we propose to play a little with the <code>gro</code> files of the membrane and the one of cisplatin. Since we have a solvent which is already equilibrated in the presence of the membrane and with the correct concentration in salt, we would like to keep it. Hence, we will split the <code>membrane.gro</code> file in two selections: the part of the solvent above the position at which we will insert the cisplatin and the rest of the system.

### 3.1. Messing with structure files

Let's start by creating two universes resp. for the membrane and for the cisplatin.

In [ ]:
import MDAnalysis as mda

# Complete the following lines
membrane = mda.Universe(XXXXX)
cisplatin = mda.Universe(XXXXX)

Now that we have created two new universes, let's find where to place cisplatin. We want to compute the free energy profile of the complete cisplatin permeation, which means that the starting configuration should be in the bulk water. Roughly speaking, bulk water starts at a cut-off distance from the lipid head groups. Let's create two selections lower and upper from the universe, based on that. To be sure that we are really in the bulk water, we can take an extra margin.

In [ ]:
# In Amber force field the cutoff distance for NB interactions is set to ??? nm. Take some margin when defining the next variable
cutoff_amber = 

# Let's base that selection on the nitrogen atom of choline groups from the upper leaflet of PC
# as PC constitute a fair amount of lipids of the outer layer
# Replace XXXXX by the correct piece of code
membrane_cog = membrane.XXXXX.XXXXX
membrane_pc_n_sel = membrane.select_atoms(f'XXXXX')
cut_z = XXXXX + XXXXX

# Let's select every molecules that are strictly below that position
lower = membrane.select_atoms(f'XXXXX prop z < {cut_z}')

# Let's select the negation of the previous selection
upper = membrane.select_atoms(f'not (XXXXX prop z < {cut_z})')
membrane.atoms.n_atoms

# Let's check that we selected all atoms
print(f'There are {membrane.atoms.n_atoms} in the system, {lower.n_atoms} in the lower part and {upper.n_atoms} in the upper part.')
if lower.n_atoms + upper.n_atoms == membrane.atoms.n_atoms:
    print('The sum of atoms in both selections gives the correct total number of atoms')
else:
    print('There is a problem!')

### 3.2. Dealing with drugs

Once we have such selections, we can shift the selection  upper. Hence, we create some space where we will insert the cisplatin. This will also avoid to have steric clashes. Choose a value of the variable shift that is appropriate (as small as possible).


In [ ]:
# Let's set that shift to 0.8 nm
shift = 8

# Let's define the translation vector
translation_vector = XXXXX

# Apply the translation vector to the upper selection
upper.translate(translation_vector)

Once we created the space to insert cisplatin molecules, we can start preparing them. Since we have a heterogeneous membrane, one way to improve the measurement of the free energy profile is to compute it multiple times at the same time. In our case, we want to place 3 molecules of cisplatin in such way that they will be as far as possible from one another.

To make it be simple, we will construct a triangle of side length equal to half of the lateral box dimension, which is centered in the center of the simulation box.

In [ ]:
# Define the size of the new universe used to place the cisplatin molecules
lateral_size = membrane.dimensions[0]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Calculate the side length of the equilateral triangle
side_length = lateral_size / 2

# Calculate the coordinates of the vertices
sqrt3 = np.sqrt(3)
A = (side_length/2, side_length*(1-sqrt3/6))
B = (3*side_length/2, side_length*(1-sqrt3/6))
C = (side_length, side_length*(1+2*sqrt3/6))

# Create a plot
plt.figure()
plt.plot([A[0], B[0], C[0], A[0],], [A[1], B[1], C[1], A[1],], marker='o', linestyle='-', color='b')

# Set axis limits
plt.xlim(0, lateral_size)
plt.ylim(0, lateral_size)

# Add labels to the vertices
plt.text(A[0], A[1], 'A', ha='right')
plt.text(B[0], B[1], 'B', ha='left')
plt.text(C[0], C[1], 'C', ha='center')

# Display the plot
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Equilateral Triangle')
plt.grid(True)

# Display the triangle
plt.show()

Now that we verified visually how the cisplatin would be placed, let's do it. Fill in the blanks of the next piece of script.

In [ ]:
# Here are the coordinates of the 3 cisplatin molecules
coordinates = np.array([
    [side_length / 2, side_length * (1 - np.sqrt(3) / 6), 0],
    [3 * side_length / 2, side_length * (1 - np.sqrt(3) / 6), 0],
    [side_length, side_length * (1 + 2 * np.sqrt(3) / 6), 0]
])

# Select cisplatin
cis_selection = XXXXX

# Translate the center of geometry to the origin and shift by shift/2 along z
cis_center = XXXXX
cis_selection.translate(-cis_center)
cis_selection.translate([0.0, 0.0, shift/2])

for i,coordinate in enumerate(coordinates):
    cis_selection.translate(XXXXX)
    if i == 0:
        ligands = mda.Merge(cis_selection)
    else:
        ligands = mda.Merge(ligands.atoms, cis_selection)
    cis_selection.translate(-XXXXX)
    
# Set the dimensions of the "ligands" universe
ligands.dimensions = XXXXX  # Set x/y dimensions to match membrane
ligands.dimensions[2] = XXXXX  # Set z dimension to XXXXX

# Save the "ligands" universe to a GRO file for visualization
ligands.atoms.write("../02.equil/ligands.gro")

### 3.3. Merging membrane and drugs together

Now that we arranged cisplatin correctly in the x/y plane, we just need to translate it and to merge <code>ligands</code> with <code>lower</code> and <code>upper</code>.

In [ ]:
# Translate ligands atoms
ligands.atoms.translate(XXXXX)

# Select all individual components to be able to save them in the same order as in the topol.top file
sel_popc = lower.select_atoms('resname POPC')
sel_psm = lower.select_atoms('resname PSM')
sel_pope = lower.select_atoms('resname POPE')
sel_sops = lower.select_atoms('resname SOPS')
sel_chl = lower.select_atoms('resname CHL')
sel_water_low = lower.select_atoms('resname SOL')
sel_water_up = upper.select_atoms('resname SOL')
sel_na_low = lower.select_atoms('resname NA')
sel_na_up = upper.select_atoms('resname NA')
sel_cl_low = lower.select_atoms('resname CL')
sel_cl_up = upper.select_atoms('resname CL')
sel_cis = ligands.select_atoms('resname CIS')

# Merge all the parts of the system into one new universe
system = mda.Merge(sel_psm, sel_popc)
system = mda.Merge(system.atoms, sel_chl)
system = mda.Merge(system.atoms, sel_pope)
system = mda.Merge(system.atoms, sel_sops)
system = mda.Merge(system.atoms, sel_na_low)
system = mda.Merge(system.atoms, sel_na_up)
system = mda.Merge(system.atoms, sel_cl_low)
system = mda.Merge(system.atoms, sel_cl_up)
system = mda.Merge(system.atoms, sel_water_low)
system = mda.Merge(system.atoms, sel_water_up)
system = mda.Merge(system.atoms, sel_cis)

# Make new selections of merged groups
sel_water = system.select_atoms('resname SOL')
sel_na = system.select_atoms('resname NA')
sel_cl = system.select_atoms('resname CL')
sel_membrane = system.select_atoms('resname POPC PSM POPE SOPS CHL CIS')
sel_solvent = system.select_atoms('resname SOL NA CL')

# Create a new index file
with mda.selections.gromacs.SelectionWriter('../02.equil/index.ndx', mode='w') as ndx:
    ndx.write(sel_popc, name = 'POPC')
    ndx.write(sel_psm, name = 'PSM')
    ndx.write(sel_pope, name = 'POPE')
    ndx.write(sel_sops, name = 'SOPS')
    ndx.write(sel_chl, name = 'CHL')
    ndx.write(sel_membrane, name = 'Membrane')
    ndx.write(sel_water, name = 'Water')
    ndx.write(sel_na, name = 'Sodium')
    ndx.write(sel_cl, name = 'Chloride')
    ndx.write(sel_solvent, name = 'Solvent')
    ndx.write(sel_cis, name = 'CIS')

# Set the dimensions of the "system" universe
system.dimensions = XXXXX  # Set x/y dimensions to match membrane
system.dimensions[2] = XXXXX

# Save the system to gro file
system.atoms.write('../02.equil/system.gro')

## 4. Relaxing the system

### 4.1. Applying restraints

Before further step, we want the system to relax and the solvent to equilibrate again. We can run a short simulation of ~20-50 ns where we are going to use <code>restraints</code> (that are external potentials applied to some degrees of freedom of a selection of atoms).

You may have seen these <code>restraints</code> called in the mdp options when you learnt how to equilibrate a membrane in the previous practicals. Indeed, by default, CHARMM-GUI proposes to gradually decrease restraints on parts of the system. As our membranes were generated with CHARMM-GUI, we will use the same kinds of restraints as they are already defined in the topology files (<code>.itp</code>).

However, not all the <code>restraints</code> are already defined. Cisplatin, which was generated separately needs to be addressed. We can do it, using <code>gmx genrestr</code>.

In [ ]:
import gromacs as gmx
from MDAnalysis.selections.gromacs import SelectionWriter

# Let's create an index file with one group being the cisplatin molecule
with mda.selections.gromacs.SelectionWriter('../02.equil/cisplatin.ndx', mode='w') as ndx:
    ndx.write(XXXXX)

# Let's generate position restraints along x, y, and z for cisplatin
gmx.genrestr(XXXXX)

Now we need to modify the topol.top file to add information regarding cisplatin:
* add the <code>include</code> instruction to point to cisplatin topology file
* add the <code>include</code> instruction to point to cisplatin position restraints file
* add the line indicating the number of cisplatin in <code>system.gro</code>

Copy to <code>02.equil</code> the following files from <code>01.input</code>:
* topol.top
* *.mdp
* toppar
* equil.sh

### 4.2. Run the equilibration

The script <code>equil.sh</code> is a direct adaption of the script provided by CHARMM-GUI when generating a membrane (in bash instead of csh). The mdp files are very close to those generated by CHARMM-GUI. The first step is an energy minimization. Then, we can simulate the system directly in the NpT ensemble (no NVT ensemble) and uses Berendsen barostat. From the second step, Parrinello-Rahman is used, although we are applying restraints. Since we do not use this simulation to make any measurements, we can avoid taking care of the Gromacs warning that pops up upon preprocessing.

**To make the practical faster, that script was already ran, and the results are in the folder <code>02.equil/results</code>.** A longer run (50 ns) without restraints on the membrane was made.

### 4.3. Checking the equilibration run

Now that we have built our system and let it relax, it is generally advisable to check whether the system reached equilibrium. A minimal check to verify if a system ran in isothermal-isobaric ensemble is equilibrated is to look at the variation of the box size.

For that use <code>gmx energy</code>.

In [ ]:
gmx.energy(XXXXX)

Let's now plot the box size for the X and the Z axes versus simulation time in two separate subplots. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load the xvg file from gmx energy
xvg_file = 'results/energy.xvg'
data = np.loadtxt(xvg_file, comments=['@', '#'])

# Separate the x and y data for Box-X and Box-Z
x = data[:, 0]
box_x = data[:, 1]
box_z = data[:, 2]

# Labels
x_label = 'Time (ps)'
y_label = '(nm)'

# Create a Matplotlib figure with two subplots (one for each dataset)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Plot the first dataset in the first subplot
ax1.plot(x, box_x, label='Box-X', linestyle='-')
ax1.set_ylabel(y_label)
ax1.set_title('GROMACS Energies')
ax1.legend()

# Plot the second dataset in the second subplot
ax2.plot(x, box_z, label='Box-Z', linestyle='-')
ax2.set_xlabel(x_label)
ax2.set_ylabel(y_label)
ax2.legend()

# Adjust the layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()

## 5. Steered MD procedure

A simulation with a gradual motion of a biasing potential applied along a CV is generally referred to as steered MD simulation. It is also common to speak of a pulling run. To compute the free energy profile using the umbrella sampling (US) procedure during the next session, we will use the CV implemented thanks to the COM pulling instructions.

Steered MD simulations can be managed with Gromacs with the <a href="https://manual.gromacs.org/documentation/current/user-guide/mdp-options.html#com-pulling">COM pulling instructions</a> set in the mdp file. We will here implement the collective variable (CV) along which we will be able to shift the biasing potential of our choice during the simulation.

We will not review all the options provided by Gromacs in the COM pulling section as this greatly exceeds the scope of this course, but we will implement a CV based on the distance (along the Z axis), which is one of the most commonly used one.

### 5.1. Flat-bottom potential

However, before implementing this collective variable, we need to take care of the 3 cisplatin molecules present in our system. As our CV will only bias the Z axis, they would be free to diffuse along X and Y. However, since we want to compute the free energy profile of permeation of one cisplatin molecule, we should be sure the molecules will not interact with one another. This can be done by applying a weak flat-bottom potential on the X/Y plane to ensure that cisplatin will be distant by at least 0.9 nm (cut-off distance).

In [ ]:
# Estimate the radius for which we do not apply any potential to cisplatin (the flat part of the flat-bottom potential)
radius = XXXXX

print(f'The radius of the flat area of the flat-bottom potential is {radius}.')

We can round this radius to 1.0 nm (because cisplatin is not punctual). Let's, once again, use restraints to implement it. In this case, we want <a href="https://manual.gromacs.org/current/reference-manual/functions/restraints.html#flat-bottomed-position-restraints">flat bottom position restraints</a> with a cylindrical geometry. The cylinder defines the region with a flat or null potential. In our case, this cylinder should be aligned to the normal to the membrane which we can safely approximate to the Z axis. 

Gromacs provides in its documentation a page detailing the syntax of every paramaters of the topology files: <a href="https://manual.gromacs.org/current/reference-manual/topologies/topology-file-formats.html#id36">format of topology related directives</a>.

You can see that the format for flat-bottom position restraints:
| Name of interaction | Topology file directive | Number of atoms | Function type | Order of parameters and their units |
| --- | --- | --- | --- | --- |
| flat-bottomed position restraint | <code>[ position_restraints  ]</code> | 1 | 2 | g, r (nm), k (kJ mol$^{-1}$ nm$^{-2}$) |

In our case, g can take, for historic reasons the values 2 or 8 (Z axis). We can use a rather weak force constant k = 500 kJ mol$^{-1}$ nm$^{-2}$ which should be enough to ensure cisplatin to not escape that cylinder, while applying a soft bias in case cisplatin diffuses in the region of the cylinder rim (this value is an educated guess, it is always advisable to run some quick check). 

Past time (section 2.1.), we applied our restraint by importing a file directly in the <code>topol.top</code> file. While this is a valid way to do it, let's make this restraint be easier to employ and allow to switch it on and off or to change the value of the force constant directly in the mdp file. Mind that we will here also avoid applying this restraint to hydrogens.

In <code>02.smd/01.rate/</code> copy the necessary files and directories from the previous simulations:
* index.ndx,
* topol.top,
* posre_cis.itp,
* step13*mdp
* toppar,
* storage/md_prod000{1.gro,2.cpt,2.tpr}.

In that folder, few modifications have to be made:
1. implementing the restraints in <code>toppar/cis0.itp</code>
2. modifying <code>topol.top</code> accordingly
3. deleting <code>posre_cis.itp</code> that was copied only to help task 1.
4. creating <code>step14_smd.mdp.template</code> from <code>step13_relaxation_solvent.mdp</code>, with the <code>COM pulling</code> section. Do not forget to edit the <code>define</code> section as well.
5. editing <code>index.ndx</code> file to add the groups linked to cisplatin. Since cisplatin is a small molecule centered around a platinum atom, we could equally defined the COG of that molecule or platinum atom to be the group used for the pulling.

In [ ]:
# Load the gro and tpr to append the index file with the pulling groups linked to Platinum atoms
system = mda.Universe(XXXXX)

sel_platinum = system.select_atoms(XXXXX)

for i,atom in enumerate(XXXXX):
    sel_single = system.select_atoms(f'XXXXX')
    print(sel_single)
    # Append index file
    with mda.selections.gromacs.SelectionWriter('index.ndx', mode='a') as ndx:
        ndx.write(sel_single, name = f'CIS{i+1}')

### 5.2. Implementing pulling code

Now we need to define the initial position of cisplatin along the (three) CV(s). For that, we can simply measure that distance using MDAnalysis.

In [ ]:
CV_init = []
for i,atom in enumerate(XXXXX):
    sel_single = system.select_atoms(f'XXXXX')
    sel_cylinder = system.select_atoms(f'XXXXX')
    CV_init.append(sel_single.positions[0][2]-sel_cylinder.center_of_geometry()[2])
print(np.mean(CV_init))

### 5.3. Selecting the rate for the SMD run

To select the rate, one can run multiple SMD simulations for different rates. Since we want to see what happens for different rates when cisplatin is inside the membrane, we need to run the SMD across 2 nm at least. The following table shows a set of SMD runs with different rates.
System | rate (ns/nm) | rate (nm/ps) |
| --- | --- | --- |
| 1 | 0.1 | 0.010000 |
| 2 | 0.2 | 0.005000 |
| 3 | 0.5 | 0.002000 |
| 4 | 1 | 0.001000 |
| 5 | 10 | 0.000100 |
| 6 | 20 | 0.000050 |
| 7 | 25 | 0.000040 |
| 8 | 40 | 0.000025 |
| 9 | 50 | 0.000020 |

Note: this test was made only for one system, for other systems, the SMD setup with optimized rate was already chosen.

Let's analyze the results:
1. Open the trajectories in VMD and use the following selections and use the command <code>rotate x by -90</code> in the <code>TCl console</code>. Does water cross the phosphorus atoms? Do you see any differences that is related to the rate of pulling? You can use the following selections:
    1. <code>resname CIS</code> (VDW)
    2. <code>same residue as water and within 6 of resname CIS</code> (Update every frame)
    3. <code>resname POPC PSM POPE SOPS and name "P.*."</code>(VDW)
2. Using <code>xmgrace</code> the reaction of the system to the applied bias, _i.e._ the measured value of the CV in the system, stored in md_pullx.xvg, vs. the position of the applied bias. Is it linear? Does the linearity depend on the rate? Is there a delay? If you notice a delay, does it depend on the rate?
3. Write a code using MDAnalysis that will analyze how cisplatin translocation perturbs the membrane (especially the head groups). Does it depend on the rate?
4. Based on these analyses, which rate would you choose? Opened question: would you like to propose complementary analyses?

In [ ]:
import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from matplotlib.colors import Normalize

# Initialize lists to store the results for each system
cv_values_list = []
num_water_near_cis_list = []
dispersion_values_list = []

# Define the number of systems
num_systems = 9

# Generate a colormap from purple to red
cmap = get_cmap('rainbow')
norm = Normalize(vmin=0, vmax=num_systems - 1)

# Loop over the systems
for i in range(1, num_systems + 1):
    tpr_file = f'smd_{i:03d}.tpr'
    xtc_file = f'smd_{i:03d}.xtc'

    system = mda.Universe(tpr_file, xtc_file)

    # Initialize lists to store the results for the current system
    cv_values = []
    num_water_near_cis = []
    dispersion_values = []

    # Loop over frames in the trajectory
    for ts in system.trajectory:
        sel_platinum = system.select_atoms('XXXXX')

        # Loop over each cisplatin molecule
        for atom in sel_platinum.molnums:
            sel_single = system.select_atoms(f'XXXXX')
            sel_cylinder = system.select_atoms(f'XXXXX')
            cv_values.append(sel_single.positions[0][2] - sel_cylinder.center_of_geometry()[2])

            sel_water = system.select_atoms(f'XXXXX')
            num_water_near_cis.append(sel_water.n_atoms)

            phosphorus_sel = system.select_atoms(f'XXXXX')
            phosphorus_z = [phosphorus.position[2] for phosphorus in phosphorus_sel.atoms if phosphorus.position[2] > sel_cylinder.center_of_geometry()[2]]
            dispersion_values.append(np.std(phosphorus_z))

    # Append the results for the current system to the lists
    cv_values_list.append(cv_values)
    num_water_near_cis_list.append(num_water_near_cis)
    dispersion_values_list.append(dispersion_values)

# Create two separate plots, each with 9 subplots
fig, axes = plt.subplots(2, 9, figsize=(18, 6))
plt.subplots_adjust(wspace=0.4, hspace=0.4)

# Plot 1: Number of water molecules near cisplatin
for i in range(num_systems):
    ax = axes[0, i]
    color = cmap(norm(i))
    ax.scatter(cv_values_list[i], num_water_near_cis_list[i], c=[color], marker='o', s=20)
    ax.set_xlabel('CV Values')
    ax.set_ylabel(f'Num Water Sys {i + 1}')
    ax.set_title(f'System {i + 1}')

# Plot 2: Standard deviation of phosphorus positions with y-axis limit of 5
for i in range(num_systems):
    ax = axes[1, i]
    color = cmap(norm(i))
    ax.scatter(cv_values_list[i], dispersion_values_list[i], c=[color], marker='o', s=20)
    ax.set_xlabel('CV Values')
    ax.set_ylabel(f'Stdev P Sys {i + 1}')
    ax.set_title(f'System {i + 1}')
    ax.set_ylim(0, 7)  # Set the y-axis limit to 0 to 5

plt.show()

## 6. Umbrella sampling

The umbrella sampling procedure consists in generating many windows along the collective variable that all start with a configuration of the system which could happen for the given value of the CV. On each of these windows, a biasing potential (generally a harmonic potential) is applied to restrain the system in the vicinity of that value, so that it could explore a small portion of the configurational space. The biasing potential for a given window is centered in a given value of the CV associated to that window.

There are a few steps to set up the umbrella sampling procedure:
1. Select the range of values of the CV to explore.
    1. Here we want we want to go from +3.5 nm to -3.5 nm.
2. Select the couple {force constant k, number of windows N$_{win}$} that we want to select.
    1. Here, we are bound by our computational resources. If we set a step of 0.1 nm (which is commonly used for ligand-membrane interactions) between two windows, we will generate 71 windows. For an all-atom run, this corresponds already to a consequent amount of ressources.
    2. To select the value of the force constant k, the best strategy is to run two adjacent windows with different values of kappa for small periods of time (_i.e._ 1 ns), and to plot the histogram of the values of the CV that are sampled to check whether there is an sufficient overlap. For such kinds of system, we can estimate that k should be in the range [200;2000] kJ mol$^{-1}$ nm$^{-2}$.
3. Select the time for which we need to sample the configurational space for each window. 
    1. Unfortunately, only an educated guess can answer that question at first. Then, only checking the convergence of the free energy profile will help. For such a small molecule as cisplatin, it was shown that 50 ns is barely enough to reach some convergence. Hence, we can start by sampling each window for 60 ns.
4. Generate the initial conformation for each window.
    1. For that, we only need to extract the frames from the SMD procedure that are the closest to the target value of the CV.
    
### 6.1. Generate the initial conformation for each window

As mentioned above, we need to generate the different frames that will associated to the set of windows for the umbrella sampling procedure. For that, we will use the trajectory produced by the SMD procedure.

Move to <code>03.us/01.dump_frames</code>.

In [ ]:
import os
import numpy as np
import pandas as pd
import MDAnalysis as mda

xvg_file_path = 'XXXXX'
xtc_file_path = 'XXXXX'
tpr_file_path = 'XXXXX'
n_windows = XXXXX
step = XXXXX
first_window = XXXXX

# Read the xvg file using Pandas, skipping the header lines
with open(xvg_file_path, 'r') as xvg_file:
    header_lines = 0
    for line in xvg_file:
        if line.startswith('@') or line.startswith('#'):
            header_lines += 1
        else:
            break

xvg_data = pd.read_csv(xvg_file_path, delim_whitespace=True, skiprows=header_lines, names=['Time (ps)', 'Bias Center (nm)'])

# Load MD trajectory using MDAnalysis
universe = mda.Universe(XXXXX)

# Create an array of window target values
window_targets = np.arange(XXXXX)

# Create a log file for recording frame information
log_file_name = 'windows.log'
with open(log_file_name, 'w') as log_file:
    log_file.write("Frame Index | Time (ps) | Bias Center (nm) | Target Value (nm)\n")

    # Store xtc timestamps and corresponding frames
    xtc_timestamps = []
    xtc_frames = []
    for ts in universe.trajectory:
        xtc_timestamps.append(ts.time)
        xtc_frames.append(ts.frame)

    for window_index, target_value in enumerate(window_targets):
        # Find the closest frame in the xvg data to the target value
        closest_frame_index = np.argmin(np.abs(xvg_data['Bias Center (nm)'] - target_value))
        closest_frame_time = xvg_data.iloc[closest_frame_index]['Time (ps)']
        closest_frame_time = float(closest_frame_time)
        
        # Find the corresponding frame in the MD trajectory
        closest_md_frame = None
        closest_time_difference = float('inf')

        # Iterate through xtc timestamps to find the corresponding frame
        for i, xtc_time in enumerate(xtc_timestamps):
            time_difference = abs(xtc_time - closest_frame_time)
            if time_difference < closest_time_difference:
                closest_md_frame = xtc_frames[i]
                closest_time_difference = time_difference

        # Debugging output
        print(f"Window {window_index + 1}/{n_windows}:")
        print(f"  Target Value: {target_value:.2f} nm")
        print(f"  Closest Frame (xvg index): {closest_frame_index}")
        print(f"  Closest Frame (MD index): {closest_md_frame}")
        print(f"  Closest Frame Time: {closest_frame_time:.2f} ps")

        # Log the information for this window
        log_line = f"{closest_md_frame:10} | {closest_frame_time:.2f} ps | {xvg_data.iloc[closest_frame_index]['Bias Center (nm)']:.2f} nm | {target_value:.2f} nm\n"
        log_file.write(log_line)

        # Save the frame as a GRO file
        gro_file_name = f'win-{window_index+1:02}.gro'
        universe.trajectory[closest_md_frame]
        universe.atoms.write(gro_file_name)

        # Print progress
        print

### 6.2. Testing different force constant

Now that we now the spacing between consecutive windows along the CV and that we have generated the initial configuration for each of these windows, we can run a quick set of simulations to undertand what values of the force constant k are acceptable for the free energy calculation.

For that, we can choose 2 windows for which the solute is in the membrane (why in the membrane?) and see for which values of k we obtain a decent overlapping of the histograms.

The results from these short runs are in the directory <code>analyze</code>. Plot the histograms for each value of the force constant in different plots. Each plot should have the histograms of windows 35 and 36.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Write that part of the code
XXXXX
        
# Define histogram parameters
hist_range = (-0.2, 0.4)
n_bins = 30

# Plot

### 6.3. Generate the windows

Move to <code>03.us/03.make_win/input</code> and create the directories for all the windows that have to be run for the US procedure.